# Day 2 — FastAPI Basics

---

Yesterday we *called* REST APIs. Today we **build one** with FastAPI — a modern Python web framework that gives us:

- **Auto-generated docs** (`/docs`, `/redoc`)
- **Type-driven validation** (powered by Pydantic)
- **Async support** for I/O-bound workloads
- One of the fastest Python frameworks in the wild (built on Starlette + uvicorn)


In [ ]:
!pip install fastapi uvicorn nest_asyncio


## Why FastAPI?

| Feature | What you get |
|---------|--------------|
| **Auto docs** | Swagger UI + ReDoc, zero config |
| **Type-driven** | Function type hints become validation + the OpenAPI schema |
| **Fast** | Built on Starlette/uvicorn — competitive with Node/Go |
| **Async-friendly** | `async def` endpoints out of the box |
| **Pydantic** | Request/response validation for free |


## Your First App

A FastAPI app is just two things: an `app` object and a function decorated with an HTTP method.


In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Demo")

@app.get("/")
def root():
    return uday()

def uday():
    return {"hello": "uday"}


## How to Run It

From a terminal, with your code in `main.py`:

```bash
uvicorn main:app --reload
```

- `main` is the module (file) name
- `app` is the FastAPI instance inside that file
- `--reload` restarts the server when code changes (dev only)

> **In Colab / Jupyter** you can't easily expose a port. Instead we'll use `TestClient` — an in-process client that hits the app directly. No server needed.


## Testing with `TestClient`

`TestClient` wraps your app in a synchronous HTTP-like interface. Same `get`/`post`/`put`/`delete` API as `requests`.


In [6]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/")
print("Status:", response.status_code)
print("Body:  ", response.json())


Status: 200
Body:   {'hello': 'world'}


## Path Parameters

Variables in the URL go in `{curly_braces}` and are passed to the function as arguments.
Add a **type hint** and FastAPI will validate it for you.


In [8]:
@app.get("/items/{item_id}")
def get_item(item_id: int):
    return {"item_id": item_id, "type": type(item_id).__name__}

print(client.get("/items/5").json())
# Bad type → 422
print(client.get("/items/uday").status_code)


{'item_id': 5, 'type': 'int'}
422


## Query Parameters

Any function argument **not** in the path is treated as a query parameter.
Give it a default to make it optional.


In [ ]:
@app.get("/items")
def list_items(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

print(client.get("/items").json())
print(client.get("/items?skip=20&limit=5").json())


## Multiple Methods on the Same Path

The same URL can have different handlers for different HTTP verbs.


In [ ]:
DB: list[dict] = []

@app.get("/notes")
def list_notes():
    return DB

@app.post("/notes")
def create_note(note: dict):
    DB.append(note)
    return {"created": note, "count": len(DB)}

print(client.post("/notes", json={"text": "hello"}).json())
print(client.get("/notes").json())


## Decorator Reference

| Decorator | HTTP method | Typical use |
|-----------|-------------|-------------|
| `@app.get(path)`    | GET    | Read a resource |
| `@app.post(path)`   | POST   | Create a resource |
| `@app.put(path)`    | PUT    | Replace a resource |
| `@app.patch(path)`  | PATCH  | Partially update a resource |
| `@app.delete(path)` | DELETE | Remove a resource |


## Async Endpoints

Declare a handler with `async def` whenever it does I/O (DB, network, file). FastAPI runs sync and async endpoints side-by-side — no special config.

```python
@app.get("/slow")
async def slow():
    await asyncio.sleep(1)
    return {"done": True}
```

**Rule of thumb:**
- `async def` for I/O-bound code (await something).
- Plain `def` for CPU-bound or blocking libraries (FastAPI runs them in a thread pool automatically).


In [11]:
import asyncio

@app.get("/slow")
async def slow():
    await asyncio.sleep(5)
    return {"done": True}

print(client.get("/slow").json())


{'done': True}


## Auto-Generated Docs

Once your server is running you get **two** interactive doc pages for free:

| URL | Tool |
|-----|------|
| `http://127.0.0.1:8000/docs`  | Swagger UI — try requests in the browser |
| `http://127.0.0.1:8000/redoc` | ReDoc — clean reference format |

The schema itself is at `/openapi.json` (used by code generators, Postman, etc).


## Recap

- `app = FastAPI()` + `@app.get(...)` is all you need to start.
- **Path parameters** go in the URL; **query parameters** are extra function args.
- Type hints **become validation** — wrong type → 422 automatically.
- Use `TestClient` for in-notebook testing; `uvicorn main:app --reload` for a real server.
- Visit `/docs` for Swagger UI — your API documents itself.
- Use `async def` for I/O; plain `def` is fine for the rest.

Next up: **Pydantic** — proper request/response models with deep validation.
